# LandIQ 2016 — unzip, attribute table, tomato codes **T15** and **T26**

**Legend (DWR):** Under **T – Truck, nursery, and berry crops**, the PDF lists **T15** (*Tomatoes, processing*) and **T26** (*Tomatoes, market*). Your attribute table can hold up to three codes per polygon in **CROPTYP1**, **CROPTYP2**, **CROPTYP3** — always scan **all** of them so you do not miss T26 in a different slot than T15.

This notebook:

1. Extracts `i15_crop_mapping_2016_shp.zip` into `data/derived/landiq_extracted/2016/` (gitignored).
2. Loads the crop shapefile with GeoPandas.
3. Summarizes columns and samples rows.
4. Finds any column containing **T15** / **T26**, then **re-checks every `CROPTYP*`** with a coverage table and a union row count.

For **other survey years**, use [`shared/explore_landiq_year.ipynb`](../shared/explore_landiq_year.ipynb) and set `YEAR` / `TOMATO_CODES` / `ZIP_FILENAME` there.

**Setup:** From the repo root: `pip install -e .` in your conda env (e.g. `gee`), then use that Jupyter kernel (or set `PYTHONPATH` to the repo root).

## 0. Paths and imports

In [ ]:
from pathlib import Path

import geopandas as gpd
import pandas as pd
from IPython.display import display

from src.landiq.legend_codes import (
    TOMATO_CODES_2016,
    attribute_table_overview,
    croptyp_column_names,
    scan_columns_for_codes,
    scan_croptyp_columns_for_codes,
    summarize_tomato_croptyp_coverage,
    tomato_mask_any_croptyp,
)
from src.landiq.zip_extract import extract_zip, find_shapefiles, pick_main_shapefile
from src.utils.paths import REPO_ROOT

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

YEAR = "2016"
RAW_YEAR = REPO_ROOT / "data" / "raw" / "landiq" / YEAR
ZIP_PATH = RAW_YEAR / "i15_crop_mapping_2016_shp.zip"
LEGEND_PDF = RAW_YEAR / "2016_DWR Standard Land Use Legend Comparison.pdf"
EXTRACT_DIR = REPO_ROOT / "data" / "derived" / "landiq_extracted" / YEAR

print("REPO_ROOT:", REPO_ROOT)
print("ZIP exists:", ZIP_PATH.is_file(), ZIP_PATH)
print("Legend PDF exists:", LEGEND_PDF.is_file(), LEGEND_PDF)
print("Extract to:", EXTRACT_DIR)
print("Tomato legend codes:", TOMATO_CODES_2016)

## 1. Unzip the shapefile archive

Run with `clear=True` to refresh extraction after re-downloading the ZIP. Sidecar files (`.dbf`, `.shx`, `.prj`, …) are written alongside the `.shp`.

In [ ]:
if not ZIP_PATH.is_file():
    raise FileNotFoundError(
        f"Missing {ZIP_PATH}. Place i15_crop_mapping_2016_shp.zip under data/raw/landiq/2016/"
    )

extract_zip(ZIP_PATH, EXTRACT_DIR, clear=True)
shp_paths = find_shapefiles(EXTRACT_DIR)
print(f"Found {len(shp_paths)} .shp file(s)")
for p in shp_paths:
    print("  ", p.relative_to(EXTRACT_DIR))

In [ ]:
shp_main = pick_main_shapefile(EXTRACT_DIR, prefer_name_contains="crop")
print("Using shapefile:", shp_main)

## 2. Load layer and attribute overview

In [ ]:
gdf = gpd.read_file(shp_main)
print("Rows:", len(gdf), "  CRS:", gdf.crs)
print("Geometry types:", gdf.geometry.geom_type.value_counts().to_dict())
gdf.head()

In [ ]:
overview = attribute_table_overview(gdf)
overview

## 3. Where do T15 and T26 appear? (full table scan)

First pass: **every** non-geometry column for exact **T15** / **T26** (trimmed). Section **3b** then focuses on **CROPTYP1–3** only, which is what you should use for filtering.

In [ ]:
hits = scan_columns_for_codes(gdf, TOMATO_CODES_2016)
if hits.empty:
    print("No exact T15/T26 matches. Try inspecting unique values in likely crop columns below.")
else:
    display(hits.sort_values(["column", "code"]))

In [ ]:
# Value counts for columns that contain tomato codes (focus work here for filtering)
if not hits.empty:
    for col in hits["column"].unique():
        print("\n===", col, "===")
        vc = gdf[col].astype(str).str.strip().value_counts(dropna=False)
        top = vc.head(40)
        display(top)
        for code in TOMATO_CODES_2016:
            if code in vc.index:
                print(f"  {code!r}: count = {vc[code]}")

## 3b. Every `CROPTYP*` slot (recommended for filtering)

Land IQ can place codes in **CROPTYP1**, **CROPTYP2**, and **CROPTYP3**. Below: counts per slot for **T15** and **T26**, then the **union** of polygons that have either code in **any** slot. Use **`landiq.crop_columns: [CROPTYP1, CROPTYP2, CROPTYP3]`** in `paths.local.yaml` with **`tomato_values: [T15, T26]`** so nothing is missed when T26 sits in a different column than T15.

In [ ]:
croptyp_cols = croptyp_column_names(gdf)
print("CROPTYP* columns:", croptyp_cols)

print("\n=== Counts per CROPTYP column × code ===")
display(summarize_tomato_croptyp_coverage(gdf, TOMATO_CODES_2016, columns=croptyp_cols))

print("\n=== Non-zero (croptyp-only scan) ===")
croptyp_hits = scan_croptyp_columns_for_codes(gdf, TOMATO_CODES_2016)
display(croptyp_hits if not croptyp_hits.empty else "(no T15/T26 in any CROPTYP slot)")

mask_any = tomato_mask_any_croptyp(gdf, TOMATO_CODES_2016, columns=croptyp_cols)
print(f"\nPolygons with T15 OR T26 in ANY listed CROPTYP slot: {int(mask_any.sum())}")

for code in TOMATO_CODES_2016:
    m = tomato_mask_any_croptyp(gdf, [code], columns=croptyp_cols)
    print(f"  {code!r} in any slot: {int(m.sum())}")

if "Crop2016" in gdf.columns:
    c = gdf["Crop2016"].astype(str).str.strip()
    tom = c.str.contains("tomato", case=False, na=False)
    print(f"\nCrop2016 text contains 'tomato' (case-insensitive): {int(tom.sum())} rows")
    print("(Sanity check vs. legend codes; labels may not match codes 1:1.)")

## 4. Likely crop / class columns (if step 3 found nothing)

Some vintages store codes with different spacing or in numeric fields. Scan column names and a few value-count previews.

In [ ]:
name_hints = ("crop", "class", "lu", "land", "comm", "dwr", "code", "sym")
likely = [c for c in gdf.columns if c != "geometry" and any(h in c.lower() for h in name_hints)]
print("Columns with name hints:", likely)
for col in likely[:12]:
    print("\n", col, "— sample values:", gdf[col].dropna().astype(str).head(8).tolist())

## 5. Optional: skim legend PDF text

Requires `pypdf` (`pip install pypdf`). Skip if not installed.

In [ ]:
try:
    from pypdf import PdfReader
except ImportError:
    print("Install pypdf to read PDF text: pip install pypdf")
else:
    if LEGEND_PDF.is_file():
        reader = PdfReader(str(LEGEND_PDF))
        text = "\n".join(page.extract_text() or "" for page in reader.pages[:5])
        for token in ("T15", "T26", "tomato", "Tomato"):
            print(token, "->", text.lower().count(token.lower()), "mentions in first 5 pages (approx)")
        print("\n--- excerpt (first 2000 chars) ---\n")
        print(text[:2000])
    else:
        print("PDF not found:", LEGEND_PDF)

## Next steps

- In `configs/paths.local.yaml` set **`landiq.tomato_values: ["T15", "T26"]`**.
- Prefer **`landiq.crop_columns: ["CROPTYP1", "CROPTYP2", "CROPTYP3"]`** so any tomato code in any slot is kept (OR logic). You can use **`landiq.crop_column: "CROPTYP2"`** only if section **3b** shows **all** T15/T26 appear solely in that column.
- Run [`02_filter_tomato_polygons.ipynb`](02_filter_tomato_polygons.ipynb) (same folder) or `python -m src.landiq.filter_tomato --input <path-to-shp>`.